In [1]:
!pip install -q -U transformers bitsandbytes peft datasets accelerate trl einops

In [2]:
base_model = "microsoft/Phi-3-mini-4k-instruct"

In [3]:
import pandas as pd

df = pd.read_parquet("hf://datasets/bkai-foundation-models/vi-alpaca/data/train-00000-of-00001-b0855b79e84114ca.parquet")

In [4]:
df = df[:4000]


In [5]:
df

,instruction,input,output
0,Hãy viết một bài blog ngắn về lợi ích của việc...,Tiêu đề: Lợi ích của việc đọc sách\nMô tả: Bài...,Bài viết: \nViệc đọc sách có rất nhiều lợi ích...
1,Tạo ra một danh sách từ vựng tiếng Anh có chủ ...,"Tiếng Anh, Toán học, Vật lý, Hóa học, Sinh học...",Danh sách từ vựng tiếng Anh với chủ đề các môn...
2,Hãy cung cấp một hướng dẫn để tạo một trình ch...,Trình chỉnh sửa video miễn phí,Hướng dẫn tạo một trình chỉnh sửa video đơn gi...
3,Chỉ ra một khả năng tích cực của người ít được...,Người bạn đề cập đến thường ít nói nhưng khi n...,"Dù ít nói, người đó thường mang lại những ý ki..."
4,Viết một hướng dẫn về cách tạo một trò chơi đơ...,"Hãy tạo một trò chơi đố vui đơn giản, nơi ngườ...",Dưới đây là một hướng dẫn về cách tạo một trò ...
...,...,...,...
3995,Viết một câu chuyện ngắn về tình bạn đáng yêu ...,,Một con chó nhỏ tên Max và một chú mèo tinh ra...
3996,Tạo một câu mô tả về cách người dùng có thể sắ...,,Để sắp xếp lại ảnh theo thứ tự ngẫu nhiên trên...
3997,Soạn thảo bức thư cảm ơn cho người bạn đã giúp...,Tên người bạn: Anh Tuấn\nCông việc bạn đã giúp...,"""Chào anh Tuấn,\n\nTôi muốn viết bức thư này đ..."
3998,Hãy tạo một câu hỏi dựa trên thông tin cung cấ...,Cuộc nổ súng ở nhà máy sản xuất tại tỉnh Bắc N...,Câu hỏi: Ngày nào xảy ra cuộc nổ súng ở nhà má...


In [7]:
conversation = []

def template_chat(ex):
    global conversation
    conversation.append([{"role": "user", "content":  ex['instruction'] + " " + ex['input']},{"role": "assistant", "content": ex['output']}])


df.apply(lambda row: template_chat(row), axis=1)

conversation[1]

[{'role': 'user',
  'content': 'Tạo ra một danh sách từ vựng tiếng Anh có chủ đề là các môn học. Tiếng Anh, Toán học, Vật lý, Hóa học, Sinh học, Lịch sử, Địa lý'},
 {'role': 'assistant',
  'content': 'Danh sách từ vựng tiếng Anh với chủ đề các môn học:\n\n1. English (Tiếng Anh)\n2. Mathematics (Toán học)\n3. Physics (Vật lý)\n4. Chemistry (Hóa học)\n5. Biology (Sinh học)\n6. History (Lịch sử)\n7. Geography (Địa lý)\n8. Literature (Văn học)\n9. Computer Science (Khoa học máy tính)\n10. Economics (Kinh tế học)\n11. Psychology (Tâm lý học)\n12. Sociology (Xã hội học)\n13. Philosophy (Triết học)\n14. Art (Mỹ thuật)\n15. Music (Âm nhạc)\n16. Physical Education (Giáo dục thể chất)\n17. Foreign Language (Ngoại ngữ)\n18. Health Education (Giáo dục sức khỏe)\n19. Government (Chính phủ)\n20. Environmental Science (Khoa học môi trường)\n\nHãy sử dụng các từ vựng này để trang bị ngôn ngữ học thuật đa dạng trong lĩnh vực giáo dục và đào tạo!'}]

In [13]:
from datasets import Dataset
dataset = Dataset.from_dict({"chat" : conversation})
dataset['chat'][1]

[{'content': 'Tạo ra một danh sách từ vựng tiếng Anh có chủ đề là các môn học. Tiếng Anh, Toán học, Vật lý, Hóa học, Sinh học, Lịch sử, Địa lý',
  'role': 'user'},
 {'content': 'Danh sách từ vựng tiếng Anh với chủ đề các môn học:\n\n1. English (Tiếng Anh)\n2. Mathematics (Toán học)\n3. Physics (Vật lý)\n4. Chemistry (Hóa học)\n5. Biology (Sinh học)\n6. History (Lịch sử)\n7. Geography (Địa lý)\n8. Literature (Văn học)\n9. Computer Science (Khoa học máy tính)\n10. Economics (Kinh tế học)\n11. Psychology (Tâm lý học)\n12. Sociology (Xã hội học)\n13. Philosophy (Triết học)\n14. Art (Mỹ thuật)\n15. Music (Âm nhạc)\n16. Physical Education (Giáo dục thể chất)\n17. Foreign Language (Ngoại ngữ)\n18. Health Education (Giáo dục sức khỏe)\n19. Government (Chính phủ)\n20. Environmental Science (Khoa học môi trường)\n\nHãy sử dụng các từ vựng này để trang bị ngôn ngữ học thuật đa dạng trong lĩnh vực giáo dục và đào tạo!',
  'role': 'assistant'}]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.padding_side = 'right'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_eos_token = True
tokenizer.add_bos_token, tokenizer.add_eos_token

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

(False, True)

In [14]:
dataset = dataset.map(lambda x: {"formatted_chat": tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [15]:
dataset['formatted_chat'][1]

'<|user|>\nTạo ra một danh sách từ vựng tiếng Anh có chủ đề là các môn học. Tiếng Anh, Toán học, Vật lý, Hóa học, Sinh học, Lịch sử, Địa lý<|end|>\n<|assistant|>\nDanh sách từ vựng tiếng Anh với chủ đề các môn học:\n\n1. English (Tiếng Anh)\n2. Mathematics (Toán học)\n3. Physics (Vật lý)\n4. Chemistry (Hóa học)\n5. Biology (Sinh học)\n6. History (Lịch sử)\n7. Geography (Địa lý)\n8. Literature (Văn học)\n9. Computer Science (Khoa học máy tính)\n10. Economics (Kinh tế học)\n11. Psychology (Tâm lý học)\n12. Sociology (Xã hội học)\n13. Philosophy (Triết học)\n14. Art (Mỹ thuật)\n15. Music (Âm nhạc)\n16. Physical Education (Giáo dục thể chất)\n17. Foreign Language (Ngoại ngữ)\n18. Health Education (Giáo dục sức khỏe)\n19. Government (Chính phủ)\n20. Environmental Science (Khoa học môi trường)\n\nHãy sử dụng các từ vựng này để trang bị ngôn ngữ học thuật đa dạng trong lĩnh vực giáo dục và đào tạo!<|end|>\n<|endoftext|>'

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,HfArgumentParser,TrainingArguments,pipeline, logging
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import torch
from trl import SFTTrainer

2024-08-21 01:24:04.351982: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-21 01:24:04.352078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-21 01:24:04.472126: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [9]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

In [10]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=500,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03
)

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage = True,
#     use_cache=True,
) 
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [16]:
import torch
print(torch.cuda.device_count())
if torch.cuda.device_count() > 1: # If more than 1 GPU
    model.is_parallelizable = True
    model.model_parallel = True

2


In [17]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field='formatted_chat',
    tokenizer=tokenizer,
    args=training_arguments,
    max_seq_length = 256,
     packing=False # Pack multiple short examples in the same input sequence to increase efficiency
)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [18]:
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


You are not running the flash-attention implementation, expect numerical differences.


Step,Training Loss
25,1.174500
50,1.003200
75,0.996600
100,0.948500
125,0.945000
150,0.959000
175,0.899400
200,0.976200
225,0.881100
250,0.898500


TrainOutput(global_step=1000, training_loss=0.8557693691253662, metrics={'train_runtime': 5079.1978, 'train_samples_per_second': 0.788, 'train_steps_per_second': 0.197, 'total_flos': 2.336025477390336e+16, 'train_loss': 0.8557693691253662, 'epoch': 1.0})

In [21]:
trainer.save_model("./kaggle/working")

In [1]:
!pip install -q -U transformers 

In [20]:
messages = [ 
    {"role": "system", "content": "Tôi là AIbot"}, 
    {"role": "user", "content": "Hãy viết một đoạn văn ngắn mô tả về quốc gia Việt Nam"}, 
] 

pipe = pipeline( 
    "text-generation", 
    model=model2, 
    tokenizer=tokenizer2,
    max_length=2048,
    do_sample=True,
    num_return_sequences=1,
    max_new_tokens=768,
    top_p=0.95,
    top_k=40,
    temperature=0.9,
    repetition_penalty=1.05,
    eos_token_id=tokenizer.eos_token_id
) 

# generation_args = { 
#     "max_new_tokens": 500, 
#     "return_full_text": False, 
#     "temperature": 0.0, 
#     "do_sample": False, 
# } 

output = pipe(messages)
#               , **generation_args) 
print(output[0]['generated_text']) 

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Both `max_new_tokens` (=768) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'system', 'content': 'Tôi là AIbot'}, {'role': 'user', 'content': 'Hãy viết một đoạn văn ngắn mô tả về quốc gia Việt Nam'}, {'role': 'assistant', 'content': ' Việt Nam, một trong những quốc gia lớn của châu Á, có biểu tượng lớn dưới sự hiện diện của nó. Lịch sử rất cao, ngoài ra thuộc truyền hình âm truyền lên bao và kỳ dịch trong xã hội sinh lý, đã góp phần quan trọng vào cuộc sống riêng của Việt Nam.\n\nÁnh để một điểm đơn giản về nguồn gốc của Việt Nam, ta cùng tôn trọng với một kỷ niệm cụ thể: lễ Đức Ông Cối Thiết. Những lễ đặc trưng trong công chúng - đầu tiên bắt đầu vào tháng 10 hoặc tháng 12 kỳ Ngọ, bắt đầu bao gồm ông Đức Ông và kỳ chào Ông Tiện - là một yêu cầu đóng góp lớn cho một loài người mới.\n\nTrong công thương của Việt Nam, văn hóa của con người có thể được tạo ra bởi sự tương tác với nhiều quốc tịch và du khách. Ví dụ, việc hiện đạt sự kỳ giám kiếm chân dung của con người Việt Nam, và các hoạt động tuyển dụng người thấy một tri thức làm hoạt động tương lai 

In [21]:
print(output[0]['generated_text'][2]['content'])

 Việt Nam, một trong những quốc gia lớn của châu Á, có biểu tượng lớn dưới sự hiện diện của nó. Lịch sử rất cao, ngoài ra thuộc truyền hình âm truyền lên bao và kỳ dịch trong xã hội sinh lý, đã góp phần quan trọng vào cuộc sống riêng của Việt Nam.

Ánh để một điểm đơn giản về nguồn gốc của Việt Nam, ta cùng tôn trọng với một kỷ niệm cụ thể: lễ Đức Ông Cối Thiết. Những lễ đặc trưng trong công chúng - đầu tiên bắt đầu vào tháng 10 hoặc tháng 12 kỳ Ngọ, bắt đầu bao gồm ông Đức Ông và kỳ chào Ông Tiện - là một yêu cầu đóng góp lớn cho một loài người mới.

Trong công thương của Việt Nam, văn hóa của con người có thể được tạo ra bởi sự tương tác với nhiều quốc tịch và du khách. Ví dụ, việc hiện đạt sự kỳ giám kiếm chân dung của con người Việt Nam, và các hoạt động tuyển dụng người thấy một tri thức làm hoạt động tương lai nhất trên Trung Quốc.

Việt Nam có một sự kế hoạch đầy sống kinh tế hấp dẫn, khoảng trắng và tạo ra một thành thạo toàn cầu. Ngoài ra, cây cánh, cánh cỏ, và hoa giày đặn là